In [ ]:
%%sql -r dataframe_1
USE DATABASE DATA5035;
USE SCHEMA SPRING26;

In [ ]:
# Paradigm: Imperative (Python)
# Each step is explicitly defined — load tables, merge, derive corridor label.

from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

# Grab the active Snowflake session — no credentials needed in notebook environment
session = get_active_session()

# Load source tables from SPRING26 schema into pandas DataFrames
road_segments  = session.table("DATA5035.SPRING26.ROAD_SEGMENTS").to_pandas()  # 103 highway segments
traffic_counts = session.table("DATA5035.SPRING26.TRAFFIC_COUNTS").to_pandas() # AADT with EV/truck share
weather_risk   = session.table("DATA5035.SPRING26.WEATHER_RISK").to_pandas()   # Weather risk per segment
incidents      = session.table("DATA5035.SPRING26.INCIDENTS").to_pandas()      # Crash/incident rates

# Merge all four tables into a single base DataFrame on shared key SEGMENT_ID
# Expected result: 103 rows with columns from all four tables
df = road_segments \
    .merge(traffic_counts, on="SEGMENT_ID") \
    .merge(weather_risk,   on="SEGMENT_ID") \
    .merge(incidents,      on="SEGMENT_ID")

# Derive corridor label — np.where is the vectorized equivalent of if/else
df["CORRIDOR"] = np.where(
    df["SEGMENT_ID"].str.contains("I70"),
    "St. Louis ↔ Kansas City",
    "St. Louis ↔ Chicago"
)

print(df)

In [ ]:
# Paradigm: Imperative (Python)
# Quick check of GEOM column format in POWER_INFRA and INTERCHANGES
# before attempting spatial joins in SQL. Confirms data is valid GeoJSON.

session = get_active_session()

power_infra  = session.table("DATA5035.SPRING26.POWER_INFRA").to_pandas()
interchanges = session.table("DATA5035.SPRING26.INTERCHANGES").to_pandas()

print(power_infra['GEOM'].head(10))
print(interchanges['GEOM'].head(10))

In [ ]:
%%sql -r dataframe_5
-- Standalone SQL version kept for reference.
-- This logic was moved into Python (see cell below) 
-- due to insufficient privileges to create a view.

/* CREATE OR REPLACE VIEW spatial_features AS
SELECT 
    rs.segment_id,
    MIN(ST_DISTANCE(rs.geom, pi.geom))                    AS dist_to_power_m,
    MIN(ST_DISTANCE(rs.geom, pi.geom)) <= 16093           AS close_to_power,
    MIN(ST_DISTANCE(rs.geom, pi.geom)) / 
        MAX(MIN(ST_DISTANCE(rs.geom, pi.geom))) OVER ()   AS power_distance_ratio,
    COUNT(DISTINCT ic.geom)                                AS interchange_count
FROM data5035.spring26.road_segments rs
LEFT JOIN data5035.spring26.power_infra  pi ON TRUE
LEFT JOIN data5035.spring26.interchanges ic ON ST_DWITHIN(rs.geom, ic.geom, 16093)
GROUP BY rs.segment_id;
*/

In [ ]:
# Paradigm: Mixed — Declarative (SQL) embedded in Imperative (Python)
# SQL handles the spatial calculations natively in Snowflake's engine.
# Python orchestrates the query execution and merges results into the base DataFrame.
# Note: Originally written as a standalone SQL view, but moved here due to
# insufficient privileges to create views in the SPRING26 schema.

spatial_query = """
    SELECT 
        rs.segment_id,

        -- Distance in meters from each segment to the nearest power source
        MIN(ST_DISTANCE(rs.geom, pi.geom))                     AS dist_to_power_m,

        -- Boolean flag: TRUE if nearest power source is within 10 miles (16,093m)
        MIN(ST_DISTANCE(rs.geom, pi.geom)) <= 16093            AS close_to_power,

        -- Normalized 0-1 ratio: 0 = closest segment to power, 1 = furthest
        MIN(ST_DISTANCE(rs.geom, pi.geom)) / 
            MAX(MIN(ST_DISTANCE(rs.geom, pi.geom))) OVER ()    AS power_distance_ratio,

        -- Count of interchanges within 10 miles of each segment
        -- ON TRUE performs a full cross join so every segment finds its nearest power source
        COUNT(DISTINCT ic.geom)                                 AS interchange_count

    FROM data5035.spring26.road_segments rs
    LEFT JOIN data5035.spring26.power_infra  pi ON TRUE
    LEFT JOIN data5035.spring26.interchanges ic ON ST_DWITHIN(rs.geom, ic.geom, 16093)
    GROUP BY rs.segment_id
"""

# Execute the spatial query in Snowflake and load results into pandas
spatial_features = session.sql(spatial_query).to_pandas()

# Merge spatial features onto the base DataFrame on SEGMENT_ID
df = df.merge(spatial_features, on="SEGMENT_ID")

# Lowercase all column names for consistent referencing in Python
df.columns = df.columns.str.lower()

print(df.shape)
print(df.columns.tolist())

In [ ]:
# Paradigm: Imperative (Python)
# Scores each segment on EV traffic volume using min-max normalization.
# Higher EV traffic = higher demand score (0 to 1).
# Min-max was chosen over percent rank to preserve proportional differences
# between segments rather than just their relative ranking position.

df['demand_score'] = (df['aadt_ev'] - df['aadt_ev'].min()) / (df['aadt_ev'].max() - df['aadt_ev'].min())

print(df[['segment_id', 'aadt_ev', 'demand_score']].sort_values('demand_score', ascending=False).head(10))

In [ ]:
# Feasibility Score

# Cap interchange count at 10
df['interchange_capped'] = df['interchange_count'].clip(upper=10)

# Normalize interchange count (min-max)
df['interchange_normalized'] = (df['interchange_capped'] - df['interchange_capped'].min()) / (df['interchange_capped'].max() - df['interchange_capped'].min())

# Invert both (lower is better for both)
df['power_score'] = 1 - df['power_distance_ratio']
df['interchange_score'] = 1 - df['interchange_normalized']

# Combine at 50/50
df['feasibility_score'] = (df['power_score'] * 0.5) + (df['interchange_score'] * 0.5)

print(df[['segment_id', 'power_distance_ratio', 'interchange_count', 'feasibility_score']].sort_values('feasibility_score', ascending=False).head(10))

In [ ]:
# Safety Score

# Normalize each component (min-max)
df['crash_normalized'] = (df['crash_rate'] - df['crash_rate'].min()) / (df['crash_rate'].max() - df['crash_rate'].min())
df['incident_normalized'] = (df['incident_rate'] - df['incident_rate'].min()) / (df['incident_rate'].max() - df['incident_rate'].min())
df['weather_normalized'] = (df['risk_score'] - df['risk_score'].min()) / (df['risk_score'].max() - df['risk_score'].min())

# Invert all three (lower is safer)
df['crash_score'] = 1 - df['crash_normalized']
df['incident_score'] = 1 - df['incident_normalized']
df['weather_score'] = 1 - df['weather_normalized']

# Combine with weights 50/30/20
df['safety_score'] = (df['incident_score'] * 0.50) + (df['crash_score'] * 0.30) + (df['weather_score'] * 0.20)

print(df[['segment_id', 'incident_rate', 'crash_rate', 'risk_score', 'safety_score']].sort_values('safety_score', ascending=False).head(10))

In [ ]:
df['lane_score'] = 1 - ((df['lanes'] - df['lanes'].min()) / (df['lanes'].max() - df['lanes'].min()))

print(df[['segment_id', 'lanes', 'lane_score']].sort_values('lane_score', ascending=False).head(10))

In [ ]:
# Weights are based on the following rationale:
# - Demand and Feasibility are equally weighted as the primary drivers (35% each)
# - Safety is a strong secondary factor (25%)
# - Lane Score is a minor tiebreaker (5%) since 77/103 segments share the minimum lane count

df['composite_score'] = (
    (df['demand_score']      * 0.35) +  # EV traffic volume
    (df['feasibility_score'] * 0.35) +  # Power proximity + interchange density
    (df['safety_score']      * 0.25) +  # Incident rate, crash rate, weather risk
    (df['lane_score']        * 0.05)    # Fewer lanes preferred for pilot installation
)

# Sort by composite score descending and display top 10 candidates
# Final 4 pilot locations will be selected from these results
# with geographic coverage considered as a manual tiebreaker
print(df[['segment_id', 'demand_score', 'feasibility_score', 'safety_score', 'lane_score', 'composite_score', 'corridor']]
      .sort_values('composite_score', ascending=False)
      .head(10))

In [ ]:
# Select top 2 segments per corridor based on composite score
top_4 = (df.groupby('corridor')
           .apply(lambda x: x.nlargest(2, 'composite_score'), include_groups=False)
           .reset_index(level=0)
           .reset_index(drop=True))

print(top_4[['segment_id', 'interstate', 'corridor', 'demand_score', 
             'feasibility_score', 'safety_score', 'lane_score', 'composite_score']])

In [ ]:
# Cell — CSV Export (Commented Out)
# This approach wrote CSVs to /tmp/ and uploaded to a Snowflake stage.
# Commented out due to insufficient privileges to access the stage.
# CSVs were instead generated by printing output and manually copying
# into GitHub via the web interface.

# df[['segment_id', 'interstate', 'corridor', 'demand_score', 
#     'feasibility_score', 'safety_score', 'lane_score', 
#     'composite_score']].sort_values('composite_score', ascending=False).to_csv('/tmp/SEGMENTS.csv', index=False)
# top_4[['segment_id', 'interstate', 'corridor', 'demand_score', 
#        'feasibility_score', 'safety_score', 'lane_score', 
#        'composite_score']].to_csv('/tmp/TOP4_SEGMENTS.csv', index=False)

# session.file.put('file:///tmp/SEGMENTS.csv', '@~/week07/', auto_compress=False, overwrite=True)
# session.file.put('file:///tmp/TOP4_SEGMENTS.csv', '@~/week07/', auto_compress=False, overwrite=True)
# print("Files saved to Snowflake stage!")

In [ ]:
print(df[['segment_id', 'interstate', 'corridor', 'demand_score', 
    'feasibility_score', 'safety_score', 'lane_score', 
    'composite_score']].sort_values('composite_score', ascending=False).to_csv(index=False))

In [ ]:
print(top_4[['segment_id', 'interstate', 'corridor', 'demand_score', 
    'feasibility_score', 'safety_score', 'lane_score', 
    'composite_score']].to_csv(index=False))